# LoRA #1 — Spy × Family style, on Kaggle

Kaggle is the reference environment (CLAUDE.md): the laptop is side research, and
Windows Smart App Control blocks the torch import chain kohya needs there anyway.

**Before running:** Settings → Accelerator → **GPU P100**, and Internet **On**.
Upload `data/datasets/spyxfamily/` as a Kaggle Dataset named `spyxfamily-panels`
(the folder with `10_sxfstyle/` inside it).

No logic lives in this notebook — it clones the repo and calls
`scripts/train_lora.py`, which builds the command from `configs/spyxfamily.yaml`.


## 1. Repo and kohya

In [ ]:
!git clone -q https://github.com/kohya-ss/sd-scripts /kaggle/working/sd-scripts
!git clone -q https://github.com/bensaadamine/manga-assistant /kaggle/working/manga-assistant
%cd /kaggle/working/manga-assistant
!git -C /kaggle/working/sd-scripts log -1 --oneline


## 2. Dependencies

Kaggle already ships a working torch for its GPU. Installing kohya's
`requirements.txt` wholesale would replace it and break CUDA, so take only the
packages the trainer needs that are not already present.

In [ ]:
!pip install -q accelerate==0.33.0 diffusers==0.25.0 transformers==4.44.2 \
    safetensors bitsandbytes lion-pytorch toml voluptuous imagesize
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")


## 3. Point the config at the Kaggle dataset

In [ ]:
from pathlib import Path
DATA = Path("/kaggle/input/spyxfamily-panels")
OUT  = Path("/kaggle/working/loras/spyxfamily")
concepts = sorted(p.name for p in DATA.glob("*_*") if p.is_dir())
n = len(list((DATA / concepts[0]).glob("*.png"))) if concepts else 0
print("concept folders:", concepts, "| images:", n)
assert concepts, "upload the dataset folder that CONTAINS 10_sxfstyle"


## 4. Check the command before spending GPU minutes

In [ ]:
!python scripts/train_lora.py --config spyxfamily.yaml \
    --data {DATA} --out {OUT} --scripts /kaggle/working/sd-scripts --dry-run


## 5. Train

~1500 steps at batch 1. Expect roughly 45–70 min on a P100. A checkpoint lands
after every epoch, so an interrupted session still leaves usable weights.

In [ ]:
!accelerate config default
!python scripts/train_lora.py --config spyxfamily.yaml \
    --data {DATA} --out {OUT} --scripts /kaggle/working/sd-scripts


## 6. Collect the weights

In [ ]:
!ls -la {OUT}
# Download spyxfamily_style.safetensors from the Output panel, then commit it
# to the DVC remote: `dvc add models/loras/spyxfamily && dvc push`


## Next

With the LoRA in hand, the headline figure of the report is **B3 vs B4** — the POC
pipeline with and without it — over the same 15 cases, plus FID against the 77
held-out panels in `data/datasets/spyxfamily_holdout/`.

If the style barely moves, the first knobs are `max_train_steps` and `alpha`, not
the dataset. If it moves too far and poses break, lower `text_encoder_lr`.